In [9]:
# libraries
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### Step 1a - Indexing (Document Ingestion)

In [29]:
video_id = 'I2ZK3ngNvvI' # only the ID, not full URL

try:
    # If you don’t care which language, this returns the “best” one
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id)
    # transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages = ['en'])
    
    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No transcript found in the requested language.")
except VideoUnavailable:
    print("The video is unavailable.")
except Exception as e:
    print("An unexpected error occurred:", str(e))

you're one of the greatest teachers of machine learning AI ever from cs231n to today what advice would you give to beginners interested in getting into machine learning beginners are often focused on like what to do and I think the focus should be more like how much you do so I I'm kind of like believer on a high level in this 10 000 hours kind of concept where you just kind of have to just pick the things where you can spend time and you you care about and you're interested in you literally have to put in 10 000 hours of work um it doesn't even like matter as much like where you put it and you'll iterate and you'll improve and you'll waste some time I don't know if there's a better way you need to put in 10 000 hours but I think it's actually really nice because I feel like there's some sense of determinism about being an expert at a thing if you spend ten thousand hours you can literally pick an arbitrary thing and I think if you spend ten thousand hours of deliberate effort and work

In [24]:
for snippet in transcript_list:
    print(snippet.text)

you're one of the greatest teachers of
machine learning AI ever from cs231n to
today what advice would you give to
beginners interested in getting into
machine learning
beginners are often focused on like what
to do and I think the focus should be
more like how much you do so I I'm kind
of like believer on a high level in this
10 000 hours kind of concept where
you just kind of have to just pick the
things where you can spend time and you
you care about and you're interested in
you literally have to put in 10 000
hours of work
um it doesn't even like matter as much
like where you put it and you'll iterate
and you'll improve and you'll waste some
time I don't know if there's a better
way you need to put in 10 000 hours but
I think it's actually really nice
because I feel like there's some sense
of determinism about being an expert at
a thing if you spend ten thousand hours
you can literally pick an arbitrary
thing and I think if you spend ten
thousand hours of deliberate effort and
work

In [31]:
transcript

"you're one of the greatest teachers of machine learning AI ever from cs231n to today what advice would you give to beginners interested in getting into machine learning beginners are often focused on like what to do and I think the focus should be more like how much you do so I I'm kind of like believer on a high level in this 10 000 hours kind of concept where you just kind of have to just pick the things where you can spend time and you you care about and you're interested in you literally have to put in 10 000 hours of work um it doesn't even like matter as much like where you put it and you'll iterate and you'll improve and you'll waste some time I don't know if there's a better way you need to put in 10 000 hours but I think it's actually really nice because I feel like there's some sense of determinism about being an expert at a thing if you spend ten thousand hours you can literally pick an arbitrary thing and I think if you spend ten thousand hours of deliberate effort and wor

In [35]:
len(transcript)

6089

### Step 1b - Indexing (Text Splitting)

In [33]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

chunks = splitter.create_documents([transcript])

In [34]:
len(chunks)

8

In [43]:
chunks[6]

Document(metadata={}, page_content="just build out a lecture that way uh sometimes I have to delete 30 minutes of content because it just went down in alley that I didn't like too much there's about a bunch of iteration and it probably takes me you know somewhere around 10 hours to create one hour of content to give one hour it's interesting I mean is it difficult to go back to the like the basics do you draw a lot of like wisdom from going back to the basics yeah going back to back propagation loss functions where they come from and one thing I like about teaching a lot honestly is it definitely strengthens your understanding uh so it's not a purely altruistic activity it's a way to learn if you have to explain something to someone uh you realize you have gaps in knowledge uh and so I even surprised myself in those lectures like also the result will obviously look at this and then the result doesn't look like it and I'm like okay I thought I understood yeah but that's why it's really 

In [41]:
chunks[6].page_content

"just build out a lecture that way uh sometimes I have to delete 30 minutes of content because it just went down in alley that I didn't like too much there's about a bunch of iteration and it probably takes me you know somewhere around 10 hours to create one hour of content to give one hour it's interesting I mean is it difficult to go back to the like the basics do you draw a lot of like wisdom from going back to the basics yeah going back to back propagation loss functions where they come from and one thing I like about teaching a lot honestly is it definitely strengthens your understanding uh so it's not a purely altruistic activity it's a way to learn if you have to explain something to someone uh you realize you have gaps in knowledge uh and so I even surprised myself in those lectures like also the result will obviously look at this and then the result doesn't look like it and I'm like okay I thought I understood yeah but that's why it's really cool to literally code you run it"

In [42]:
chunks[6].metadata

{}

### Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [47]:
embeddings = OpenAIEmbeddings(model= "text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [48]:
vector_store.index_to_docstore_id

{0: 'd9527a10-528c-4ae0-a8cf-0596888651bb',
 1: 'a3a7a511-52bd-4aa7-abe6-1af77a0ee2bf',
 2: '68af8e48-e4fd-4753-a335-cfa340ad4621',
 3: 'f971e8af-e66b-4f61-bc9a-e6be3491c204',
 4: '8e5a1795-09b2-4026-a5a7-8c3988bf18d9',
 5: '026ea499-a111-4b0a-9628-688035f24627',
 6: 'c931a597-5a72-4b38-8f6a-592abdbf3344',
 7: '78a4f1f0-410e-484c-9a6b-3e3506e94390'}

In [49]:
vector_store.get_by_ids(['8e5a1795-09b2-4026-a5a7-8c3988bf18d9'])

[Document(id='8e5a1795-09b2-4026-a5a7-8c3988bf18d9', metadata={}, page_content="uh what do you love about teaching you seem to find yourself often in the like drawn to teaching you're very good at it but you're also drawn to it I mean I don't think I love teaching I love happy humans and happy humans like when I teach yes I I wouldn't say I hate teaching I tolerate teaching but it's not like the act of teaching that I like it's it's that um you know I I have some I have something I'm actually okay at it yes I'm okay at teaching and people appreciate it a lot yeah and uh so I'm just happy to try to be helpful and uh teaching itself is not like the most I mean it's really no it can be really annoying frustrating I was working on a bunch of lectures just now I was reminded back to my days of 231 and just how much work it is to create some of these materials and make them good the amount of iteration and thought and you go down blind alleys and just how much you change it so creating somet

### Step 2 - Retrieval

In [50]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [51]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000282F70E3310>, search_kwargs={'k': 4})

In [52]:
retriever.invoke('how he feels about teaching')

[Document(id='8e5a1795-09b2-4026-a5a7-8c3988bf18d9', metadata={}, page_content="uh what do you love about teaching you seem to find yourself often in the like drawn to teaching you're very good at it but you're also drawn to it I mean I don't think I love teaching I love happy humans and happy humans like when I teach yes I I wouldn't say I hate teaching I tolerate teaching but it's not like the act of teaching that I like it's it's that um you know I I have some I have something I'm actually okay at it yes I'm okay at teaching and people appreciate it a lot yeah and uh so I'm just happy to try to be helpful and uh teaching itself is not like the most I mean it's really no it can be really annoying frustrating I was working on a bunch of lectures just now I was reminded back to my days of 231 and just how much work it is to create some of these materials and make them good the amount of iteration and thought and you go down blind alleys and just how much you change it so creating somet

### Step 3 - Augmentation

In [53]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.2)

In [54]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.
    
    {context}
    Question: {question}
    """,
    input_variables= ['context', 'question']
)

In [55]:
question = "what he feels abour teaching?"
retrieved_docs = retriever.invoke(question)

In [56]:
retrieved_docs

[Document(id='8e5a1795-09b2-4026-a5a7-8c3988bf18d9', metadata={}, page_content="uh what do you love about teaching you seem to find yourself often in the like drawn to teaching you're very good at it but you're also drawn to it I mean I don't think I love teaching I love happy humans and happy humans like when I teach yes I I wouldn't say I hate teaching I tolerate teaching but it's not like the act of teaching that I like it's it's that um you know I I have some I have something I'm actually okay at it yes I'm okay at teaching and people appreciate it a lot yeah and uh so I'm just happy to try to be helpful and uh teaching itself is not like the most I mean it's really no it can be really annoying frustrating I was working on a bunch of lectures just now I was reminded back to my days of 231 and just how much work it is to create some of these materials and make them good the amount of iteration and thought and you go down blind alleys and just how much you change it so creating somet

In [59]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

context_text

"uh what do you love about teaching you seem to find yourself often in the like drawn to teaching you're very good at it but you're also drawn to it I mean I don't think I love teaching I love happy humans and happy humans like when I teach yes I I wouldn't say I hate teaching I tolerate teaching but it's not like the act of teaching that I like it's it's that um you know I I have some I have something I'm actually okay at it yes I'm okay at teaching and people appreciate it a lot yeah and uh so I'm just happy to try to be helpful and uh teaching itself is not like the most I mean it's really no it can be really annoying frustrating I was working on a bunch of lectures just now I was reminded back to my days of 231 and just how much work it is to create some of these materials and make them good the amount of iteration and thought and you go down blind alleys and just how much you change it so creating something good um in terms of like educational value is really hard and uh it's not\

In [60]:
final_prompt = prompt.invoke({"context":context_text, "question":question})

In [61]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context is insufficient, just say you don't know.\n\n    uh what do you love about teaching you seem to find yourself often in the like drawn to teaching you're very good at it but you're also drawn to it I mean I don't think I love teaching I love happy humans and happy humans like when I teach yes I I wouldn't say I hate teaching I tolerate teaching but it's not like the act of teaching that I like it's it's that um you know I I have some I have something I'm actually okay at it yes I'm okay at teaching and people appreciate it a lot yeah and uh so I'm just happy to try to be helpful and uh teaching itself is not like the most I mean it's really no it can be really annoying frustrating I was working on a bunch of lectures just now I was reminded back to my days of 231 and just how much work it is to create some of these materials and make them good the amount 

### STEP - 4 Generation

In [62]:
answer = llm.invoke(final_prompt)
print(answer.content)

He doesn't love teaching itself; he tolerates it. He enjoys seeing happy humans and feels that teaching helps create that happiness. He acknowledges that teaching can be frustrating and difficult, but he appreciates being able to help others and finds value in the process of teaching as it strengthens his own understanding.


## Building a Chain

In [63]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [71]:
def format_doc(retrived_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrived_docs)
    return context_text

In [72]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_doc),
    'question' : RunnablePassthrough()
})

In [73]:
parallel_chain.invoke('What are advice to beginners he has given')

{'context': "you're one of the greatest teachers of machine learning AI ever from cs231n to today what advice would you give to beginners interested in getting into machine learning beginners are often focused on like what to do and I think the focus should be more like how much you do so I I'm kind of like believer on a high level in this 10 000 hours kind of concept where you just kind of have to just pick the things where you can spend time and you you care about and you're interested in you literally have to put in 10 000 hours of work um it doesn't even like matter as much like where you put it and you'll iterate and you'll improve and you'll waste some time I don't know if there's a better way you need to put in 10 000 hours but I think it's actually really nice because I feel like there's some sense of determinism about being an expert at a thing if you spend ten thousand hours you can literally pick an arbitrary thing and I think if you spend ten thousand hours of deliberate ef

In [74]:
parser = StrOutputParser()

In [75]:
main_chain = parallel_chain | prompt | llm | parser

In [76]:
main_chain.invoke('Can you summarize the video')

'The video discusses the challenges and processes involved in creating educational content, particularly in the context of teaching complex topics like backpropagation and loss functions in machine learning. The speaker emphasizes the iterative nature of content creation, noting that it often requires multiple takes and significant editing to produce engaging and informative lectures. They highlight the importance of coding and practical demonstrations in teaching, as it helps solidify understanding and reveals gaps in knowledge. The speaker also reflects on the value of learning from mistakes and experiences, suggesting that past challenges contribute to future growth and intuition in the field. Overall, the speaker expresses a passion for teaching, viewing it as a valuable learning experience rather than purely an altruistic endeavor.'